In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)

claims = pd.read_csv('claims_merged.csv')
print("Shape:", claims.shape)

print("\duplicate rows:", claims.duplicated().sum())
print("Duplicate ClaimID:", claims['ClaimID'].duplicated().sum())

nulls = claims.isnull().sum()
print("\nNull columns:")
print(nulls[nulls > 0].sort_values(ascending=False))

<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\kk\AppData\Local\Temp\ipykernel_18908\548569677.py:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  print("\duplicate rows:", claims.duplicated().sum())
C:\Users\kk\AppData\Local\Temp\ipykernel_18908\548569677.py:4: DtypeWarning: Columns (0: AdmissionDt, 1: DischargeDt, 2: DiagnosisGroupCode, 3: DOD) have mixed types. Specify dtype option on import or set low_memory=False.
  claims = pd.read_csv('claims_merged.csv')


Shape: (558211, 59)
\duplicate rows: 0
Duplicate ClaimID: 0

Null columns:
ClmProcedureCode_6       558211
ClmProcedureCode_5       558202
ClmProcedureCode_4       558093
ClmProcedureCode_3       557242
DOD                      554080
ClmDiagnosisCode_10      553201
ClmProcedureCode_2       552721
ClmProcedureCode_1       534901
DiagnosisGroupCode       517737
DischargeDt              517737
AdmissionDt              517737
Days                     517737
Cost_Per_Day             517737
ClmDiagnosisCode_9       516396
ClmDiagnosisCode_8       504767
ClmDiagnosisCode_7       492034
ClmDiagnosisCode_6       473819
ClmDiagnosisCode_5       446287
OperatingPhysician       443764
ClmAdmitDiagnosisCode    412312
ClmDiagnosisCode_4       393675
OtherPhysician           358475
ClmDiagnosisCode_3       315156
ClmDiagnosisCode_2       195606
ClmDiagnosisCode_1        10453
AttendingPhysician         1508
DeductibleAmtPaid           899
dtype: int64


In [4]:
date_cols = ['ClaimStartDt', 'ClaimEndDt', 'AdmissionDt', 'DischargeDt', 'DOB', 'DOD']

for c in date_cols:
    claims[c] = pd.to_datetime(claims[c], errors='coerce')

print(claims[date_cols].dtypes)

print("\nMissing dates:")
print(claims[date_cols].isnull().sum())

print("\nClaimEnd before ClaimStart (invalid):", (claims['ClaimEndDt'] < claims['ClaimStartDt']).sum())
print("Discharge before Admission (invalid):", (claims['DischargeDt'] < claims['AdmissionDt']).sum())

ClaimStartDt    datetime64[us]
ClaimEndDt      datetime64[us]
AdmissionDt     datetime64[us]
DischargeDt     datetime64[us]
DOB             datetime64[us]
DOD             datetime64[us]
dtype: object

Missing dates:
ClaimStartDt         0
ClaimEndDt           0
AdmissionDt     517737
DischargeDt     517737
DOB                  0
DOD             554080
dtype: int64

ClaimEnd before ClaimStart (invalid): 0
Discharge before Admission (invalid): 0


In [5]:
before = len(claims)

claims = claims.drop_duplicates()
claims = claims.drop_duplicates(subset='ClaimID', keep='first')

print("Rows before:", before)
print("Rows after :", len(claims))
print("Removed    :", before - len(claims))

Rows before: 558211
Rows after : 558211
Removed    : 0


In [6]:
bad_claim = claims['ClaimEndDt'] < claims['ClaimStartDt']
bad_stay  = claims['DischargeDt'] < claims['AdmissionDt']

print("Invalid claim dates:", bad_claim.sum())
print("Invalid stay dates :", bad_stay.sum())

claims = claims[~(bad_claim | bad_stay)].copy()
print("Rows now:", len(claims))

Invalid claim dates: 0
Invalid stay dates : 0
Rows now: 558211


In [7]:
# Outpatient claims have no deductible, so missing means 0
claims['DeductibleAmtPaid'] = claims['DeductibleAmtPaid'].fillna(0)

# Physician columns: missing means unknown
phys_cols = ['AttendingPhysician', 'OperatingPhysician', 'OtherPhysician']
for c in phys_cols:
    claims[c] = claims[c].fillna('UNKNOWN')

# Diagnosis and procedure codes: missing means none recorded
code_cols = [c for c in claims.columns if c.startswith('ClmDiagnosisCode_')
             or c.startswith('ClmProcedureCode_')
             or c in ['ClmAdmitDiagnosisCode', 'DiagnosisGroupCode']]
for c in code_cols:
    claims[c] = claims[c].fillna('NONE')

remaining = claims.isnull().sum()
print(remaining[remaining > 0])

AdmissionDt     517737
DischargeDt     517737
DOD             554080
Days            517737
Cost_Per_Day    517737
dtype: int64


In [8]:
# Is the patient dead (has a death date)?
claims['IsDead'] = claims['DOD'].notnull().astype(int)

# Claim made after patient's death (used for phantom billing rule)
claims['ClaimAfterDeath'] = (claims['ClaimStartDt'] > claims['DOD']).astype(int)

# Days admitted (inpatient only, outpatient stays NaN)
claims['AdmitDays'] = (claims['DischargeDt'] - claims['AdmissionDt']).dt.days + 1

# Claim duration in days
claims['ClaimDays'] = (claims['ClaimEndDt'] - claims['ClaimStartDt']).dt.days + 1

# Patient age at claim time
claims['Age'] = ((claims['ClaimStartDt'] - claims['DOB']).dt.days / 365.25).round(0)

print("Claims after death:", claims['ClaimAfterDeath'].sum())
print(claims[['IsDead', 'ClaimAfterDeath', 'AdmitDays', 'ClaimDays', 'Age']].describe())

Claims after death: 0
              IsDead  ClaimAfterDeath     AdmitDays      ClaimDays  \
count  558211.000000         558211.0  40474.000000  558211.000000   
mean        0.007400              0.0      6.665168       2.727940   
std         0.085707              0.0      5.638538       4.904984   
min         0.000000              0.0      1.000000       1.000000   
25%         0.000000              0.0      3.000000       1.000000   
50%         0.000000              0.0      5.000000       1.000000   
75%         0.000000              0.0      8.000000       1.000000   
max         1.000000              0.0     36.000000      37.000000   

                 Age  
count  558211.000000  
mean       73.298629  
std        13.021629  
min        25.000000  
25%        68.000000  
50%        74.000000  
75%        82.000000  
max       101.000000  


In [9]:
claims.to_csv('claims_clean.csv', index=False)
print("Saved:", claims.shape)

Saved: (558211, 64)
